# PyKIS API 사용 가이드

이 노트북은 PyKIS (Python Korea Investment Securities) API의 주요 기능을 소개하고 사용 방법을 설명합니다.

## 목차
1. 환경 설정 및 초기화
2. 계좌 정보 조회
3. 주식 시세 조회
4. 조건 검색
5. 투자자 정보 조회
6. WebSocket 실시간 데이터

최종 업데이트: 2025-01-21

## 1. 환경 설정 및 초기화

In [ ]:
# 필요한 라이브러리 import
import os
import sys
import json
import pandas as pd
from datetime import datetime, timedelta

# PyKIS 경로 추가 (필요한 경우)
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))

# PyKIS 모듈 import
from pykis import Agent
from pykis.core.client import KISClient
from pykis.stock.api import StockAPI
from pykis.account.api import AccountAPI
from pykis.stock.condition import ConditionAPI
from pykis.stock.investor import InvestorAPI
from pykis.websocket.client import KisWebSocket

print("PyKIS 모듈 로드 완료!")

### Agent 초기화

In [ ]:
# Agent 초기화 (환경 변수에서 인증 정보 자동 로드)
agent = Agent()
print(f"Agent 초기화 완료")
print(f"모드: {'모의투자' if agent.is_paper else '실전투자'}")
print(f"계좌번호: {agent.account_info.get('CANO', 'N/A')}")

## 2. 계좌 정보 조회

In [ ]:
# 계좌 잔고 조회
balance = agent.get_account_balance()
if balance:
    print("=== 계좌 잔고 정보 ===")
    print(f"예수금: {balance.get('예수금', 0):,}원")
    print(f"총평가금액: {balance.get('총평가금액', 0):,}원")
    print(f"총손익: {balance.get('총손익', 0):,}원")
    print(f"수익률: {balance.get('수익률', 0):.2f}%")
else:
    print("계좌 잔고 조회 실패")

In [ ]:
# 보유 주식 조회
stocks = agent.get_stock_balance()
if stocks and len(stocks) > 0:
    print(f"\n=== 보유 주식 ({len(stocks)}종목) ===")
    for stock in stocks[:5]:  # 상위 5개만 표시
        print(f"- {stock.get('prdt_name', 'N/A')}: {stock.get('hldg_qty', 0):,}주")
        print(f"  평가금액: {stock.get('evlu_amt', 0):,}원")
        print(f"  수익률: {stock.get('evlu_pfls_rt', 0):.2f}%")
else:
    print("보유 주식이 없습니다.")

## 3. 주식 시세 조회

In [ ]:
# 삼성전자 현재가 조회
stock_code = "005930"  # 삼성전자
price_info = agent.get_stock_price(stock_code)

if price_info is not None and not price_info.empty:
    print(f"=== {stock_code} 현재가 정보 ===")
    info = price_info.iloc[0]
    print(f"현재가: {info.get('stck_prpr', 0):,}원")
    print(f"전일대비: {info.get('prdy_vrss', 0):,}원 ({info.get('prdy_ctrt', 0)}%)")
    print(f"거래량: {info.get('acml_vol', 0):,}주")
    print(f"시가: {info.get('stck_oprc', 0):,}원")
    print(f"고가: {info.get('stck_hgpr', 0):,}원")
    print(f"저가: {info.get('stck_lwpr', 0):,}원")
else:
    print("시세 조회 실패")

In [ ]:
# 일봉 데이터 조회
daily_data = agent.get_daily_price(stock_code, period="D")

if daily_data is not None and not daily_data.empty:
    print(f"\n=== {stock_code} 최근 일봉 데이터 ===")
    # 최근 5일 데이터 표시
    for idx, row in daily_data.head(5).iterrows():
        print(f"{row.get('stck_bsop_date', '')}:")
        print(f"  종가: {row.get('stck_clpr', 0):,}원")
        print(f"  거래량: {row.get('acml_vol', 0):,}주")
else:
    print("일봉 데이터 조회 실패")

In [ ]:
# 호가 정보 조회
orderbook = agent.get_orderbook(stock_code)

if orderbook is not None:
    print(f"\n=== {stock_code} 호가 정보 ===")
    
    # 매도 호가 (상위 3개)
    print("매도 호가:")
    for i in range(1, 4):
        price = orderbook.get(f'askp{i}', 0)
        qty = orderbook.get(f'askp_rsqn{i}', 0)
        if price:
            print(f"  {price:,}원: {qty:,}주")
    
    print("\n매수 호가:")
    for i in range(1, 4):
        price = orderbook.get(f'bidp{i}', 0)
        qty = orderbook.get(f'bidp_rsqn{i}', 0)
        if price:
            print(f"  {price:,}원: {qty:,}주")
else:
    print("호가 정보 조회 실패")

## 4. 조건 검색

In [ ]:
# 조건검색 결과 조회
condition_stocks = agent.get_condition_stocks(user_id="unohee", seq=0)

if condition_stocks:
    print(f"=== 조건검색 결과 ({len(condition_stocks)}종목) ===")
    for stock in condition_stocks[:10]:  # 상위 10개만 표시
        code = stock.get('code', '')
        name = stock.get('name', '')
        if code and name:
            print(f"- {code}: {name}")
else:
    print("조건검색 결과가 없습니다.")

## 5. 투자자 정보 조회

In [ ]:
# 투자자별 매매동향 조회
investor_data = agent.get_stock_investor(stock_code)

if investor_data is not None and not investor_data.empty:
    print(f"=== {stock_code} 투자자별 매매동향 ===")
    data = investor_data.iloc[0]
    
    # 개인
    print(f"개인:")
    print(f"  순매수수량: {data.get('개인_순매수수량', 0):,}주")
    print(f"  순매수금액: {data.get('개인_순매수거래대금_억', 0):.2f}억원")
    
    # 외국인
    if 'frgn_ntby_qty' in data:
        print(f"\n외국인:")
        print(f"  순매수수량: {int(data.get('frgn_ntby_qty', 0)):,}주")
    
    # 기관
    if 'orgn_ntby_qty' in data:
        print(f"\n기관:")
        print(f"  순매수수량: {int(data.get('orgn_ntby_qty', 0)):,}주")
else:
    print("투자자 정보 조회 실패")

In [ ]:
# 거래원 정보 조회
member_data = agent.get_stock_member(stock_code)

if member_data:
    print(f"\n=== {stock_code} 거래원 정보 ===")
    output = member_data.get('output', {})
    
    # 매수 상위 거래원
    print("매수 상위 거래원:")
    for i in range(1, 4):
        member_name = output.get(f'shnu_mbcr_name{i}', '')
        qty = output.get(f'total_shnu_qty{i}', '0')
        if member_name:
            print(f"  {i}. {member_name}: {int(qty):,}주")
    
    # 매도 상위 거래원
    print("\n매도 상위 거래원:")
    for i in range(1, 4):
        member_name = output.get(f'seln_mbcr_name{i}', '')
        qty = output.get(f'total_seln_qty{i}', '0')
        if member_name:
            print(f"  {i}. {member_name}: {int(qty):,}주")
else:
    print("거래원 정보 조회 실패")

## 6. WebSocket 실시간 데이터

In [ ]:
# WebSocket 클라이언트 초기화 (실시간 데이터)
import asyncio
import nest_asyncio
nest_asyncio.apply()  # Jupyter에서 asyncio 사용을 위해

# WebSocket 연결 설정
ws_client = KisWebSocket(agent.client)

# 실시간 데이터 수신 콜백 함수
def on_message(data):
    """실시간 데이터 수신 시 호출되는 콜백"""
    print(f"실시간 데이터 수신: {data.get('header', {}).get('tr_id', 'Unknown')}")
    if 'body' in data:
        body = data['body']
        print(f"  종목: {body.get('mksc_shrn_iscd', '')}")
        print(f"  현재가: {body.get('stck_prpr', '')}")
        print(f"  체결량: {body.get('cntg_vol', '')}")

# 콜백 설정
ws_client.on_message = on_message

print("WebSocket 클라이언트 초기화 완료")
print("실시간 데이터를 수신하려면 connect() 후 subscribe()를 호출하세요.")

In [ ]:
# WebSocket 연결 및 구독 예시 (실행 시 주의)
async def test_websocket():
    """WebSocket 테스트 함수"""
    try:
        # 연결
        await ws_client.connect()
        print("WebSocket 연결 성공")
        
        # 실시간 체결가 구독
        await ws_client.subscribe(
            tr_type="H0STCNT0",  # 실시간 체결가
            tr_key=stock_code     # 종목코드
        )
        print(f"{stock_code} 실시간 체결가 구독 시작")
        
        # 10초간 데이터 수신
        await asyncio.sleep(10)
        
        # 구독 해제
        await ws_client.unsubscribe(
            tr_type="H0STCNT0",
            tr_key=stock_code
        )
        print("구독 해제")
        
        # 연결 종료
        await ws_client.close()
        print("WebSocket 연결 종료")
        
    except Exception as e:
        print(f"WebSocket 오류: {e}")

# 실행 (장중에만 데이터 수신 가능)
# await test_websocket()
print("WebSocket 테스트 준비 완료")
print("장중에 test_websocket()을 실행하면 실시간 데이터를 수신할 수 있습니다.")

## 7. 주문 기능 (실전 투자 시 주의)

In [ ]:
# 주문 가능 수량 조회
order_cash = agent.get_cash_available(stock_code, price=50000)

if order_cash:
    print(f"=== 주문 가능 정보 ===")
    print(f"주문 가능 금액: {order_cash.get('ord_psbl_cash', 0):,}원")
    print(f"주문 가능 수량: {order_cash.get('max_buy_qty', 0):,}주")
else:
    print("주문 가능 정보 조회 실패")

In [ ]:
# 매수 주문 예시 (실행하지 마세요!)
"""
# 실제 주문 코드 (주석 처리됨)
order_result = agent.buy(
    ticker=stock_code,
    quantity=1,
    price=50000,  # 지정가
    order_type="00"  # 00: 지정가, 01: 시장가
)

if order_result:
    print(f"주문 성공: {order_result}")
else:
    print("주문 실패")
"""

print("주문 기능은 실전 투자 시 신중하게 사용하세요.")
print("테스트는 모의투자 환경에서 진행하는 것을 권장합니다.")

## 8. 유용한 유틸리티 함수들

In [ ]:
# KOSPI200 선물 코드 생성
from pykis.stock.api import get_kospi200_futures_code

futures_code = get_kospi200_futures_code()
print(f"현재 거래 중인 KOSPI200 선물 코드: {futures_code}")

# 선물 코드는 만기월에 따라 자동으로 계산됩니다
# 예: 101W03 (3월물), 101W06 (6월물), 101W09 (9월물), 101W12 (12월물)

In [ ]:
# 휴일 여부 확인
today = datetime.now().strftime("%Y%m%d")
is_holiday = agent.is_holiday(today)

if is_holiday is not None:
    print(f"오늘({today})은 {'휴일' if is_holiday else '영업일'}입니다.")
else:
    print("휴일 정보 조회 실패")

In [ ]:
# 시장 변동성 조회
market_info = agent.get_market_fluctuation()

if market_info:
    print("=== 시장 변동성 정보 ===")
    print(json.dumps(market_info, indent=2, ensure_ascii=False)[:500])  # 일부만 표시
else:
    print("시장 정보 조회 실패")

## 9. 데이터 저장 및 분석

In [ ]:
# DataFrame으로 데이터 정리 및 저장
if daily_data is not None and not daily_data.empty:
    # 일봉 데이터를 CSV로 저장
    filename = f"{stock_code}_daily_{datetime.now().strftime('%Y%m%d')}.csv"
    daily_data.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"일봉 데이터가 {filename}에 저장되었습니다.")
    
    # 간단한 통계 분석
    print("\n=== 일봉 데이터 통계 ===")
    print(daily_data[['stck_clpr', 'acml_vol']].describe())

In [ ]:
# 여러 종목 한 번에 조회
stock_list = ["005930", "000660", "035720"]  # 삼성전자, SK하이닉스, 카카오
stock_prices = {}

print("=== 복수 종목 시세 조회 ===")
for code in stock_list:
    price_df = agent.get_stock_price(code)
    if price_df is not None and not price_df.empty:
        price = price_df.iloc[0]['stck_prpr']
        stock_prices[code] = price
        print(f"{code}: {price:,}원")

# DataFrame으로 변환
if stock_prices:
    prices_df = pd.DataFrame(list(stock_prices.items()), columns=['종목코드', '현재가'])
    print("\n=== 가격 DataFrame ===")
    print(prices_df)

## 10. 마무리

In [ ]:
print("=" * 50)
print("PyKIS API 사용 가이드 완료")
print("=" * 50)
print("\n주요 기능:")
print("1. Agent 클래스를 통한 통합 API 접근")
print("2. 계좌 정보 및 잔고 조회")
print("3. 실시간 시세 및 호가 정보")
print("4. 조건 검색 기능")
print("5. 투자자별 매매 동향")
print("6. WebSocket 실시간 데이터")
print("7. 주문 기능 (buy/sell)")
print("\n자세한 내용은 PyKIS 문서를 참조하세요.")